In [1]:
import fitz
import json
import re
import pandas as pd
import google.generativeai as genai
import ast
import time
from pathlib import Path
from tqdm import tqdm

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/fg/60gnk2vx4x99rxrxc7tbh99r0000gn/T/ipykernel_54829/1535180140.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(Path(".env"))

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel("gemini-3.1-flash-lite")

In [3]:
KEYWORDS = [
    "biodiversity",
    "ecosystem",
    "species",
    "habitat",
    "forest",
    "wetland",
    "flora",
    "fauna",
    "restoration",
    "conservation",
    "afforestation",
    "reforestation",
    "wildlife",
    "ecological",
    "pollution",
    "land restoration",
    "mine closure",
    "water stewardship",
    "environmental impact"
]


def biodiversity_page(text):

    text_lower = text.lower()

    return any(k in text_lower for k in KEYWORDS)

In [4]:
def clean_text(text):

    text = text.replace("\x00", " ")

    text = re.sub(r'[•▪●◦·]', ' ', text)

    text = re.sub(r'\.{3,}', ' ', text)

    text = re.sub(r'[-_=]{3,}', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [5]:
def extract_pages(pdf_path):

    doc = fitz.open(pdf_path)

    pages = []

    for i, page in enumerate(doc):

        text = page.get_text()

        pages.append({
            "page_number": i + 1,
            "text": text
        })

    return pages

In [6]:
EXTRACTION_PROMPT = """
You are an ESG biodiversity analyst.

Extract ONLY statements directly related to biodiversity,
ecology, environmental impacts on nature, ecosystem
restoration, conservation, or biodiversity management.

Relevant topics include:
- biodiversity
- ecosystems
- habitats
- forests
- wetlands
- ecological restoration
- afforestation
- reforestation
- species conservation
- pollution affecting biodiversity
- land degradation
- mine restoration
- ecological protection
- groundwater
- freshwater
- watershed management
- ecological rehabilitation
- biodiversity management plans
- native vegetation restoration
- wildlife conservation

IMPORTANT GEOGRAPHIC RULE:

Extract statements ONLY if they refer to:

1. Operations, projects, mines, plants, facilities,
   restoration activities, conservation initiatives,
   biodiversity management plans, or environmental
   performance within India.

2. Company-wide disclosures that are clearly intended
   to describe the company's operations in India.

DO NOT extract statements that refer exclusively to:
- foreign subsidiaries
- overseas operations
- projects outside India
- international conservation programs
- biodiversity initiatives conducted entirely outside India

If a statement does not explicitly mention a country
but appears to describe the company's core operations,
mines, plants, manufacturing facilities, restoration
activities, or environmental programs, assume it refers
to Indian operations and keep it.

DO NOT extract:
- social welfare
- CSR unrelated to biodiversity
- women empowerment
- generic sustainability claims
- finance
- DEI initiatives
- education
- drinking water access programs
- generic agriculture
- climate discussion unrelated to ecosystems
- international case studies unrelated to Indian operations

IMPORTANT:
Return ONLY VALID JSON.

OUTPUT FORMAT:

[
  {
    "statement": "..."
  }
]

If no relevant statements exist:
return []

NO markdown.
NO explanations.
ONLY JSON.
"""

In [7]:
ANALYSIS_PROMPT = """
You are an ESG biodiversity analyst.

Analyze the biodiversity statement.

------------------------------------------------

1. CLASSIFICATION

Allowed values ONLY:

- Goal
- Commitment
- SMART Target

Definitions:

SMART Target:
specific, measurable, achievable,
relevant, and time-bound.

Commitment:
specific intention lacking one or more
SMART elements.

Goal:
broad aspirational statement lacking
metrics or timelines.

------------------------------------------------

2. SMART EVALUATION

Return ONLY true or false for:
- Specific
- Measurable
- Achievable
- Relevant
- TimeBound

------------------------------------------------

3. GBF TARGET MAPPING

Allowed GBF targets ONLY:

1  = Spatial_Planning
2  = Ecosystem_Restoration
3  = Protected_Areas_30x30
4  = Species_Genetic_Diversity
5  = Sustainable_Wild_Species_Use
6  = Invasive_Species
7  = Pollution_Reduction
8  = Climate_Impact_Minimization
9  = Sustainable_Human_Use
10 = Sustainable_Agriculture_Forestry
11 = Ecosystem_Services
12 = Urban_Biodiversity
13 = Genetic_Resource_Sharing
14 = Biodiversity_Decision_Making
15 = Business_Disclosure_Risk
16 = Sustainable_Consumption
17 = Biosafety_Governance
18 = Harmful_Subsidy_Reform
19 = Biodiversity_Finance
20 = Capacity_Technology
21 = Biodiversity_Knowledge_Data
22 = Inclusive_Participation_Justice
23 = Gender_Equality

Use ONLY integers from this list.

------------------------------------------------

IMPORTANT:
Return ONLY VALID JSON.

OUTPUT FORMAT:

{
  "statement": "...",

  "classification": "Commitment",

  "SMART": {
    "Specific": true,
    "Measurable": false,
    "Achievable": true,
    "Relevant": true,
    "TimeBound": false
  },

  "GBF_targets": [2, 7]
}

NO markdown.
ONLY JSON.
"""

In [8]:
GENERATION_CONFIG = {
    "temperature": 0,
    "max_output_tokens": 512,
    "response_mime_type": "application/json"
}

MAX_RETRIES = 5


def _generate(prompt, label):
    """Call the model, retrying transient API failures.

    Returns the response text, or None if every attempt failed
    (the caller skips that page/statement rather than aborting
    the whole run)."""

    for attempt in range(MAX_RETRIES):

        try:
            response = model.generate_content(
                prompt,
                generation_config=GENERATION_CONFIG
            )

            # Warn instead of silently losing data.
            try:
                reason = response.candidates[0].finish_reason.name
                if reason == "MAX_TOKENS":
                    print(
                        f"\nWARNING: {label} hit max_output_tokens "
                        f"- output truncated, JSON likely invalid"
                    )
            except (AttributeError, IndexError):
                pass

            return response.text

        except Exception as e:

            last = attempt == MAX_RETRIES - 1

            wait = 2 ** attempt

            print(
                f"\n{label}: {type(e).__name__} "
                f"(attempt {attempt + 1}/{MAX_RETRIES})"
                + ("" if last else f", retrying in {wait}s")
            )
            print(f"  {e}")

            if last:
                print(
                    f"{label}: giving up after "
                    f"{MAX_RETRIES} attempts - skipping"
                )
                return None

            time.sleep(wait)


def extract_statements(page_text):

    return _generate(
        EXTRACTION_PROMPT + "\n\n" + page_text,
        "extraction"
    )


def analyze_statement(statement):

    return _generate(
        ANALYSIS_PROMPT + "\n\n" + statement,
        "analysis"
    )

In [9]:
def validate_json(output):

    if output is None:
        return None

    cleaned = output.strip()

    cleaned = cleaned.replace("```json", "")
    cleaned = cleaned.replace("```", "")

    try:
        return json.loads(cleaned)
    except Exception:
        try:
            return ast.literal_eval(cleaned)
        except Exception as e:
            print("\nINVALID JSON")
            print(cleaned)
            print(e)
            return None

In [10]:
def normalize_extraction(extracted):

    normalized = []

    seen = set()

    def add_statement(text):

        if not isinstance(text, str):
            return

        text = text.strip()

        if len(text) < 20:
            return

        if text in seen:
            return

        seen.add(text)

        normalized.append({
            "statement": text
        })

    # CASE 1
    # extracted = {"statements": [...]}

    if isinstance(extracted, dict):

        # A
        if "statements" in extracted:

            statements = extracted["statements"]

            if isinstance(statements, list):

                for item in statements:

                    # dict item
                    if isinstance(item, dict):

                        if "statement" in item:

                            add_statement(
                                item["statement"]
                            )

                    # raw string
                    elif isinstance(item, str):

                        add_statement(item)

        # B
        elif "statement" in extracted:

            add_statement(extracted["statement"])

    # CASE 2
    # extracted = [...]

    elif isinstance(extracted, list):

        for item in extracted:

            # dict
            if isinstance(item, dict):

                if "statement" in item:

                    add_statement(item["statement"])

            # raw string
            elif isinstance(item, str):

                add_statement(item)

    return normalized

In [11]:
pdf_folder = Path("Data/reports")
output_folder = Path("Data/final_outputs")
output_folder.mkdir(parents=True, exist_ok=True)

zero_folder = Path("Data/0 statement csvs")

already_done = (
    {f.stem for f in output_folder.glob("*.csv")} |
    {f.stem for f in zero_folder.glob("*.csv")}
)

for pdf_file in sorted(pdf_folder.glob("*.pdf")):

    if pdf_file.stem in already_done:
        continue

    print(f"\n{'='*60}")
    print(f"Processing: {pdf_file.name}")
    print(f"{'='*60}")

    # Fresh results for THIS company only
    results = []

    pages = extract_pages(pdf_file)

    for page in tqdm(pages):

        text = clean_text(page["text"])

        if len(text) < 100:
            continue

        extraction_output = extract_statements(text)

        extracted = validate_json(extraction_output)

        if extracted is None:
            continue

        extracted = normalize_extraction(extracted)

        for item in extracted:

            statement = item["statement"]

            analysis_output = analyze_statement(statement)

            analyzed = validate_json(analysis_output)

            if analyzed is None:
                continue

            analyzed["company"] = pdf_file.stem
            analyzed["page"] = page["page_number"]

            results.append(analyzed)

    # -------------------------
    # Save this company's CSV
    # -------------------------

    company_df = pd.DataFrame(
        results,
        columns=[
            "statement",
            "classification",
            "SMART",
            "GBF_targets",
            "company",
            "page"
        ]
    )

    output_file = output_folder / f"{pdf_file.stem}.csv"

    company_df.to_csv(
        output_file,
        index=False
    )

    print(
        f"\nSaved {len(company_df)} rows to {output_file}"
    )


Processing: Bombay Wire Ropes Ltd.pdf


100%|███████████████████████████████████████████| 66/66 [01:06<00:00,  1.00s/it]


Saved 1 rows to final_outputs/Bombay Wire Ropes Ltd.csv
